<a href="https://colab.research.google.com/github/Arjx01/ML2_Arjun-M_4NI23CI019/blob/main/ML_2_5b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import math
def foil_gain(p0, n0, p1, n1):
    if p1 == 0:
        return float("-inf")
    if p0 == 0:
        return float("-inf")
    if p0 + n0 == 0:
        return float("-inf")
    if p1 + n1 == 0:
        return float("-inf")
    old_probability = p0 / (p0 + n0)
    new_probability = p1 / (p1 + n1)
    if old_probability <= 0:
        return float("-inf")
    if new_probability <= 0:
        return float("-inf")
    return p1 * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )
def satisfies(example, rule):
    for attribute, value in rule:
        if str(example[attribute]) != str(value):
            return False
    return True
def covered_examples(df, rule):
    if len(rule) == 0:
        return df.copy()
    mask = df.apply(
        lambda row: satisfies(row, rule),
        axis=1
    )
    return df[mask].copy()
def generate_candidate_literals(
    df,
    rule,
    features
):
    used_attributes = {
        attribute
        for attribute, value in rule
    }
    candidates = []
    for feature in features:
        if feature in used_attributes:
            continue
        values = df[feature].dropna().unique()
        for value in values:
            candidates.append(
                (feature, value)
            )
    return candidates
def rule_to_string(rule):
    if len(rule) == 0:
        return "Survived = 1"
    conditions = []
    for attribute, value in rule:
        conditions.append(
            f"{attribute} = {value}"
        )
    return (
        "IF "
        + " AND ".join(conditions)
        + " THEN Survived = 1"
    )
def learn_one_rule(
    df,
    target,
    positive_value,
    negative_value,
    features
):
    rule = []
    while True:
        covered = covered_examples(
            df,
            rule
        )
        positives = covered[
            covered[target] == positive_value
        ]
        negatives = covered[
            covered[target] == negative_value
        ]
        if len(negatives) == 0:
            if len(positives) > 0:
                return rule
            return None
        p0 = len(positives)
        n0 = len(negatives)
        candidates = generate_candidate_literals(
            df,
            rule,
            features
        )
        if len(candidates) == 0:
            return None
        best_literal = None
        best_gain = float("-inf")
        for literal in candidates:
            new_rule = rule + [literal]
            new_covered = covered_examples(
                df,
                new_rule
            )
            p1 = len(
                new_covered[
                    new_covered[target]
                    == positive_value
                ]
            )
            n1 = len(
                new_covered[
                    new_covered[target]
                    == negative_value
                ]
            )
            gain = foil_gain(
                p0,
                n0,
                p1,
                n1
            )
            if gain > best_gain:
                best_gain = gain
                best_literal = literal
        if best_literal is None:
            return None
        rule.append(best_literal)
        print(
            "      Added:",
            best_literal,
            "| Gain:",
            round(best_gain, 4)
        )
        print(
            "      Rule:",
            rule_to_string(rule)
        )
        if len(rule) >= len(features):
            break
    return rule
def foil(
    df,
    target,
    positive_value,
    negative_value,
    features
):
    positives_remaining = df[
        df[target] == positive_value
    ].copy()
    negatives = df[
        df[target] == negative_value
    ].copy()
    learned_rules = []
    rule_number = 1
    print("\n====================================")
    print("             FOIL")
    print("====================================")
    print(
        "\nInitial positive examples:",
        len(positives_remaining)
    )
    print(
        "Initial negative examples:",
        len(negatives)
    )
    while len(positives_remaining) > 0:
        print(
            f"\n\n========== LEARNING RULE "
            f"{rule_number} =========="
        )
        print(
            "Positive examples remaining:",
            len(positives_remaining)
        )
        training_data = pd.concat(
            [
                positives_remaining,
                negatives
            ]
        )
        new_rule = learn_one_rule(
            training_data,
            target,
            positive_value,
            negative_value,
            features
        )
        if new_rule is None:
            print(
                "\nCould not find another rule."
            )
            break
        learned_rules.append(
            new_rule
        )
        print(
            "\nLearned:",
            rule_to_string(new_rule)
        )
        covered_positive = covered_examples(
            positives_remaining,
            new_rule
        )
        print(
            "Positive examples covered:",
            len(covered_positive)
        )
        positives_remaining = (
            positives_remaining.drop(
                covered_positive.index
            )
        )
        rule_number += 1
    return learned_rules
def predict(example, rules):
    for rule in rules:
        if satisfies(example, rule):
            return 1
    return 0
def preprocess_titanic(df):
    df = df.copy()
    df["Sex"] = (
        df["Sex"]
        .fillna("Unknown")
        .astype(str)
        .str.lower()
    )
    df["Pclass"] = (
        df["Pclass"]
        .fillna(0)
        .astype(str)
    )
    df["SibSp"] = (
        df["SibSp"]
        .fillna(0)
        .astype(str)
    )
    df["Parch"] = (
        df["Parch"]
        .fillna(0)
        .astype(str)
    )
    df["Embarked"] = (
        df["Embarked"]
        .fillna("Unknown")
        .astype(str)
    )
    df["Age"] = pd.to_numeric(
        df["Age"],
        errors="coerce"
    )
    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[
            -np.inf,
            12,
            18,
            30,
            50,
            np.inf
        ],
        labels=[
            "Child",
            "Teen",
            "YoungAdult",
            "Adult",
            "Senior"
        ]
    )
    df["AgeGroup"] = (
        df["AgeGroup"]
        .astype(object)
        .fillna("Unknown")
    )
    df["Fare"] = pd.to_numeric(
        df["Fare"],
        errors="coerce"
    )
    df["FareGroup"] = pd.cut(
        df["Fare"],
        bins=[
            -np.inf,
            10,
            30,
            100,
            np.inf
        ],
        labels=[
            "Cheap",
            "Medium",
            "Expensive",
            "VeryExpensive"
        ]
    )
    df["FareGroup"] = (
        df["FareGroup"]
        .astype(object)
        .fillna("Unknown")
    )
    return df
if __name__ == "__main__":
    df = pd.read_csv("train.csv")
    print("Original dataset:")
    print(df.head())
    print(
        "\nDataset shape:",
        df.shape
    )
    df = preprocess_titanic(df)
    features = [
        "Sex",
        "Pclass",
        "SibSp",
        "Parch",
        "Embarked",
        "AgeGroup",
        "FareGroup"
    ]
    target = "Survived"
    rules = foil(
        df=df,
        target=target,
        positive_value=1,
        negative_value=0,
        features=features
    )
    print("\n\n====================================")
    print("          FINAL FOIL RULES")
    print("====================================")
    for i, rule in enumerate(
        rules,
        start=1
    ):
        print(
            f"\nRule {i}:"
        )
        print(
            " ",
            rule_to_string(rule)
        )
    predictions = []
    for _, row in df.iterrows():
        prediction = predict(
            row,
            rules
        )
        predictions.append(
            prediction
        )
    df["Prediction"] = predictions
    accuracy = (
        df["Prediction"]
        == df[target]
    ).mean()
    print(
        "\n===================================="
    )
    print(
        "Training Accuracy:",
        round(accuracy * 100, 2),
        "%"
    )
    print(
        "===================================="
    )

Original dataset:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500  